In [ ]:
"""
AS/400 Phase 1 – Transform & Aufbereitung
==========================================
Liest die Raw-Excel (aus as400_phase1_raw_extract.py),
führt alle Berechnungen, Joins und Aggregationen in Python durch
und speichert eine Analyse-Excel, die direkt vom Notebook
as400_phase1_analyse.ipynb geladen werden kann.

Voraussetzungen:
    pip install pandas openpyxl

Nutzung:
    python as400_phase1_transform.py as400_raw_YYYYMMDD_HHMM.xlsx
    (ohne Argument: aktuellste as400_raw_*.xlsx wird automatisch genommen)
"""

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime
import sys
import glob
import os


# ─────────────────────────────────────────────
# Input-Datei ermitteln
# ─────────────────────────────────────────────
def find_raw_file():
    if len(sys.argv) > 1:
        f = sys.argv[1]
        if not os.path.exists(f):
            print(f"✗ Datei nicht gefunden: {f}")
            sys.exit(1)
        return f
    files = sorted(glob.glob("as400_raw_*.xlsx"), reverse=True)
    if not files:
        print("✗ Keine as400_raw_*.xlsx gefunden. Bitte zuerst raw_extract ausführen.")
        sys.exit(1)
    print(f"→ Verwende: {files[0]}")
    return files[0]


def load_raw(raw_file):
    print(f"Lade Raw-Daten aus: {raw_file} ...")
    sheets = pd.read_excel(raw_file, sheet_name=None, dtype=str)
    # Numerische Spalten sauber konvertieren (AS/400 liefert manchmal Strings)
    numeric_cols = {
        "raw_systables":    ["COLUMN_COUNT"],
        "raw_systablestat": ["NUMBER_ROWS", "DATA_SIZE", "OVERFLOW"],
        "raw_syscolumns":   ["ORDINAL_POSITION", "LENGTH", "NUMERIC_SCALE"],
        "raw_sysindexes":   ["NUMBER_OF_COLUMNS"],
        "raw_syskeys":      ["ORDINAL_POSITION"],
        "raw_syscolumnstat":["DISTINCT_COUNT", "NULL_COUNT"],
    }
    for sheet, num_cols in numeric_cols.items():
        if sheet in sheets:
            for col in num_cols:
                if col in sheets[sheet].columns:
                    sheets[sheet][col] = pd.to_numeric(sheets[sheet][col], errors="coerce")
    for name, df in sheets.items():
        print(f"  {name:<25} {len(df):>7,} Zeilen")
    return sheets


# ─────────────────────────────────────────────
# TRANSFORM-FUNKTIONEN
# Jede Funktion erzeugt ein aufbereitetes DataFrame.
# Gibt es eine Raw-View nicht → leeres DataFrame mit Hinweis.
# ─────────────────────────────────────────────

def t_schema_uebersicht(raw):
    """Aggregation auf Schema-Ebene: Anzahl Tabellen, Views, Aliases."""
    df = raw.get("raw_systables", pd.DataFrame())
    if df.empty:
        return df
    grp = df.groupby("TABLE_SCHEMA")
    result = pd.DataFrame({
        "schema_name":      grp["TABLE_NAME"].count(),
        "anzahl_gesamt":    grp["TABLE_NAME"].count(),
        "tabellen":         grp["TABLE_TYPE"].apply(lambda x: (x == "T").sum()),
        "views":            grp["TABLE_TYPE"].apply(lambda x: (x == "V").sum()),
        "aliases":          grp["TABLE_TYPE"].apply(lambda x: (x == "A").sum()),
        "physisch":         grp["TABLE_TYPE"].apply(lambda x: (x == "P").sum()),
    }).reset_index(drop=True)
    # schema_name aus Index wiederherstellen
    result.insert(0, "schema_name", grp["TABLE_NAME"].count().index)
    result = result.drop(columns=["schema_name"], errors="ignore")
    # sauberer Weg:
    agg = (
        df.assign(
            ist_tabelle = df["TABLE_TYPE"].eq("T").astype(int),
            ist_view    = df["TABLE_TYPE"].eq("V").astype(int),
            ist_alias   = df["TABLE_TYPE"].eq("A").astype(int),
            ist_physisch= df["TABLE_TYPE"].eq("P").astype(int),
        )
        .groupby("TABLE_SCHEMA")
        .agg(
            anzahl_gesamt  = ("TABLE_NAME",  "count"),
            tabellen       = ("ist_tabelle",  "sum"),
            views          = ("ist_view",     "sum"),
            aliases        = ("ist_alias",    "sum"),
            physisch       = ("ist_physisch", "sum"),
        )
        .reset_index()
        .rename(columns={"TABLE_SCHEMA": "schema_name"})
        .sort_values("anzahl_gesamt", ascending=False)
    )
    return agg


def t_tabellen(raw):
    """Tabellen angereichert mit Statistik (Zeilen, Größe)."""
    tabs = raw.get("raw_systables", pd.DataFrame())
    stat = raw.get("raw_systablestat", pd.DataFrame())
    if tabs.empty:
        return tabs

    t = tabs[tabs["TABLE_TYPE"].isin(["T", "P"])].copy()
    t = t.rename(columns={
        "TABLE_SCHEMA":           "schema_name",
        "TABLE_NAME":             "tabelle",
        "TABLE_TYPE":             "typ",
        "TABLE_TEXT":             "beschreibung",
        "COLUMN_COUNT":           "spalten_anzahl",
        "LAST_ALTERED_TIMESTAMP": "letzte_aenderung",
        "SYSTEM_TABLE_NAME":      "systemname",
    })
    keep = ["schema_name","tabelle","typ","beschreibung","spalten_anzahl","letzte_aenderung","systemname"]
    t = t[[c for c in keep if c in t.columns]]

    if not stat.empty:
        s = stat.rename(columns={
            "TABLE_SCHEMA": "schema_name",
            "TABLE_NAME":   "tabelle",
            "NUMBER_ROWS":  "zeilen_anzahl",
            "DATA_SIZE":    "daten_bytes",
        })
        s["daten_kb"] = s["daten_bytes"].fillna(0) / 1024
        s["daten_mb"] = s["daten_bytes"].fillna(0) / 1024 / 1024
        s = s[["schema_name","tabelle","zeilen_anzahl","daten_kb","daten_mb"]]
        t = t.merge(s, on=["schema_name","tabelle"], how="left")
        t["zeilen_anzahl"] = t["zeilen_anzahl"].fillna(0).astype(int)
        t["daten_kb"]      = t["daten_kb"].fillna(0).round(2)
        t["daten_mb"]      = t["daten_mb"].fillna(0).round(4)

    return t.sort_values(["schema_name","tabelle"])


def t_spalten(raw):
    """Spalten mit lesbaren Spaltennamen."""
    df = raw.get("raw_syscolumns", pd.DataFrame())
    if df.empty:
        return df
    rename = {
        "TABLE_SCHEMA":      "schema_name",
        "TABLE_NAME":        "tabelle",
        "ORDINAL_POSITION":  "position",
        "COLUMN_NAME":       "spalte",
        "SYSTEM_COLUMN_NAME":"system_spalte",
        "DATA_TYPE":         "datentyp",
        "LENGTH":            "laenge",
        "NUMERIC_SCALE":     "dezimalstellen",
        "IS_NULLABLE":       "nullable",
        "COLUMN_DEFAULT":    "standard_wert",
        "COLUMN_TEXT":       "beschreibung",
        "HAS_DEFAULT":       "hat_default",
    }
    cols_present = {k: v for k, v in rename.items() if k in df.columns}
    return df.rename(columns=cols_present)[list(cols_present.values())].sort_values(
        ["schema_name","tabelle","position"]
    )


def t_views(raw):
    """Views mit Beschreibung."""
    views = raw.get("raw_sysviews", pd.DataFrame())
    tabs  = raw.get("raw_systables", pd.DataFrame())
    if views.empty:
        return views
    v = views.rename(columns={
        "TABLE_SCHEMA":    "schema_name",
        "TABLE_NAME":      "view_name",
        "VIEW_DEFINITION": "view_definition",
    })
    if not tabs.empty:
        t = tabs[["TABLE_SCHEMA","TABLE_NAME","TABLE_TEXT","COLUMN_COUNT"]].rename(columns={
            "TABLE_SCHEMA": "schema_name",
            "TABLE_NAME":   "view_name",
            "TABLE_TEXT":   "beschreibung",
            "COLUMN_COUNT": "spalten_anzahl",
        })
        v = v.merge(t, on=["schema_name","view_name"], how="left")
    return v.sort_values(["schema_name","view_name"])


def t_indizes(raw):
    """Indizes mit lesbaren Namen."""
    df = raw.get("raw_sysindexes", pd.DataFrame())
    if df.empty:
        return df
    rename = {
        "TABLE_SCHEMA":      "schema_name",
        "TABLE_NAME":        "tabelle",
        "INDEX_NAME":        "index_name",
        "IS_UNIQUE":         "eindeutig",
        "INDEX_TYPE":        "index_typ",
        "NUMBER_OF_COLUMNS": "anzahl_spalten",
        "SYSTEM_INDEX_NAME": "system_name",
        "INDEX_TEXT":        "beschreibung",
    }
    cols_present = {k: v for k, v in rename.items() if k in df.columns}
    return df.rename(columns=cols_present)[list(cols_present.values())].sort_values(
        ["schema_name","tabelle","index_name"]
    )


def t_schluessel(raw):
    """Schlüsselspalten."""
    df = raw.get("raw_syskeys", pd.DataFrame())
    if df.empty:
        return df
    rename = {
        "TABLE_SCHEMA":    "schema_name",
        "TABLE_NAME":      "tabelle",
        "INDEX_NAME":      "index_name",
        "COLUMN_NAME":     "spalte",
        "ORDINAL_POSITION":"position",
        "ORDERING":        "sortierung",
    }
    cols_present = {k: v for k, v in rename.items() if k in df.columns}
    return df.rename(columns=cols_present)[list(cols_present.values())].sort_values(
        ["schema_name","tabelle","index_name","position"]
    )


def t_tabellenstatistik(raw):
    """Tabellenstatistik bereinigt und sortiert."""
    df = raw.get("raw_systablestat", pd.DataFrame())
    if df.empty:
        return df
    rename = {
        "TABLE_SCHEMA":         "schema_name",
        "TABLE_NAME":           "tabelle",
        "NUMBER_ROWS":          "zeilen_anzahl",
        "DATA_SIZE":            "daten_bytes",
        "OVERFLOW":             "overflow_zeilen",
        "LAST_USED_TIMESTAMP":  "zuletzt_genutzt",
        "STATISTICS_TIMESTAMP": "statistik_stand",
    }
    cols_present = {k: v for k, v in rename.items() if k in df.columns}
    r = df.rename(columns=cols_present)[list(cols_present.values())].copy()
    r["daten_mb"] = r["daten_bytes"].fillna(0) / 1024 / 1024
    r["daten_mb"] = r["daten_mb"].round(3)
    r["zeilen_anzahl"] = r["zeilen_anzahl"].fillna(0).astype(int)
    return r.sort_values("zeilen_anzahl", ascending=False)


def t_spaltenstatistik(raw):
    """Spaltenstatistik bereinigt."""
    df = raw.get("raw_syscolumnstat", pd.DataFrame())
    if df.empty:
        return df
    rename = {
        "TABLE_SCHEMA":         "schema_name",
        "TABLE_NAME":           "tabelle",
        "COLUMN_NAME":          "spalte",
        "DISTINCT_COUNT":       "eindeutige_werte",
        "NULL_COUNT":           "null_anzahl",
        "STATISTICS_TIMESTAMP": "statistik_stand",
    }
    cols_present = {k: v for k, v in rename.items() if k in df.columns}
    return df.rename(columns=cols_present)[list(cols_present.values())].sort_values(
        ["schema_name","tabelle","spalte"]
    )


def t_programme(raw):
    """Programme/Routinen."""
    df = raw.get("raw_sysroutines", pd.DataFrame())
    if df.empty:
        return df
    rename = {
        "ROUTINE_SCHEMA":         "schema_name",
        "ROUTINE_NAME":           "programm",
        "ROUTINE_TYPE":           "typ",
        "EXTERNAL_NAME":          "externer_name",
        "LANGUAGE":               "sprache",
        "ROUTINE_TEXT":           "beschreibung",
        "LAST_ALTERED_TIMESTAMP": "letzte_aenderung",
        "CREATED":                "erstellt",
    }
    cols_present = {k: v for k, v in rename.items() if k in df.columns}
    return df.rename(columns=cols_present)[list(cols_present.values())].sort_values(
        ["schema_name","programm"]
    )


def t_wichtigkeit_score(tabs_df, stat_df, idx_df):
    """
    Gewichteter Wichtigkeits-Score pro Tabelle.
    Wird vom Notebook für die Heatmap verwendet.
    Score = 0.5 * Zeilen(norm) + 0.3 * Indizes(norm) + 0.2 * Spalten(norm)
    """
    if tabs_df.empty:
        return pd.DataFrame()

    t = tabs_df[["schema_name","tabelle"]].copy()
    if "spalten_anzahl" in tabs_df.columns:
        t["spalten_anzahl"] = pd.to_numeric(tabs_df["spalten_anzahl"], errors="coerce").fillna(0)
    else:
        t["spalten_anzahl"] = 0

    if not stat_df.empty and "zeilen_anzahl" in stat_df.columns:
        s = stat_df[["schema_name","tabelle","zeilen_anzahl","daten_mb"]].copy()
        t = t.merge(s, on=["schema_name","tabelle"], how="left")
    else:
        t["zeilen_anzahl"] = 0
        t["daten_mb"]      = 0
    t["zeilen_anzahl"] = t["zeilen_anzahl"].fillna(0)
    t["daten_mb"]      = t["daten_mb"].fillna(0) if "daten_mb" in t.columns else 0

    if not idx_df.empty and "tabelle" in idx_df.columns:
        i = idx_df.groupby(["schema_name","tabelle"]).size().reset_index(name="index_anzahl")
        t = t.merge(i, on=["schema_name","tabelle"], how="left")
    else:
        t["index_anzahl"] = 0
    t["index_anzahl"] = t["index_anzahl"].fillna(0)

    for col in ["zeilen_anzahl","spalten_anzahl","index_anzahl"]:
        mx = t[col].max()
        t[f"{col}_norm"] = (t[col] / mx).round(4) if mx > 0 else 0.0

    t["wichtigkeit_score"] = (
        t["zeilen_anzahl_norm"]  * 0.5 +
        t["index_anzahl_norm"]   * 0.3 +
        t["spalten_anzahl_norm"] * 0.2
    ).round(4)

    return t.sort_values("wichtigkeit_score", ascending=False)


# ─────────────────────────────────────────────
# Excel schreiben
# ─────────────────────────────────────────────

def style_sheet(ws, tab_color):
    header_fill = PatternFill("solid", fgColor="1F4E79")
    header_font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    alt_fill    = PatternFill("solid", fgColor="EBF3FB")
    data_font   = Font(name="Arial", size=10)
    thin        = Side(style="thin", color="D0D0D0")
    border      = Border(left=thin, right=thin, top=thin, bottom=thin)
    left        = Alignment(horizontal="left",   vertical="center")
    center      = Alignment(horizontal="center", vertical="center")

    ws.sheet_properties.tabColor = tab_color
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 20

    for cell in ws[1]:
        cell.font      = header_font
        cell.fill      = header_fill
        cell.border    = border
        cell.alignment = center

    for row_idx, row in enumerate(ws.iter_rows(min_row=2), 2):
        fill = alt_fill if row_idx % 2 == 0 else None
        for cell in row:
            cell.font      = data_font
            cell.border    = border
            cell.alignment = left
            if fill:
                cell.fill = fill

    for col_idx in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_idx)
        vals = [ws.cell(r, col_idx).value for r in range(1, min(60, ws.max_row) + 1)]
        max_len = max((len(str(v)) if v is not None else 0 for v in vals), default=8)
        ws.column_dimensions[col_letter].width = min(max_len + 4, 50)


TAB_COLORS = ["2E75B6","00B050","7030A0","FF8C00","C00000","4BACC6","F79646","0070C0","70AD47","1F4E79"]

SHEET_DESCRIPTIONS = {
    "00_Schema_Übersicht":   "Aggregation pro Schema/Library",
    "01_Tabellen":           "Tabellen + Statistik (Zeilen, MB)",
    "02_Spalten":            "Alle Spalten aller Tabellen",
    "03_Views":              "Views inkl. Definition",
    "04_Indizes":            "Alle Indizes",
    "05_Schlüssel":          "Index-Schlüsselspalten",
    "06_Tabellenstatistik":  "Rohe Statistik (Zeilen, Größe)",
    "07_Spaltenstatistik":   "Distinct-Werte & NULL-Anteile",
    "08_Programme":          "Programme / Routinen",
    "09_Wichtigkeit_Score":  "Gewichteter Score für Heatmap",
}


def write_info_sheet(wb, output_file, raw_file, transforms):
    ws = wb.create_sheet("00_Info", 0)
    ws.sheet_properties.tabColor = "1F4E79"
    ws["A1"] = "AS/400 Phase 1 – Aufbereitete Analyse-Daten"
    ws["A1"].font = Font(name="Arial", bold=True, size=14, color="1F4E79")
    ws["A3"] = "Erstellt am:"
    ws["B3"] = datetime.now().strftime("%d.%m.%Y %H:%M")
    ws["A4"] = "Quelle (Raw):"
    ws["B4"] = raw_file
    ws["A5"] = "Ausgabe:"
    ws["B5"] = output_file
    for r in range(3, 6):
        ws.cell(r, 1).font = Font(name="Arial", bold=True, size=10)
        ws.cell(r, 2).font = Font(name="Arial", size=10)

    hf = PatternFill("solid", fgColor="1F4E79")
    hfont = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    for col, label in zip(["A","B","C"], ["SHEET","BESCHREIBUNG","DATENSÄTZE"]):
        c = ws[f"{col}7"]
        c.value = label
        c.font  = hfont
        c.fill  = hf
        c.alignment = Alignment(horizontal="center")

    for i, (sheet, desc) in enumerate(SHEET_DESCRIPTIONS.items(), 8):
        df = transforms.get(sheet, pd.DataFrame())
        ws.cell(i, 1, sheet).font = Font(name="Arial", size=10)
        ws.cell(i, 2, desc).font  = Font(name="Arial", size=10)
        ws.cell(i, 3, len(df)).font = Font(name="Arial", size=10)
        ws.cell(i, 3).alignment = Alignment(horizontal="right")

    ws.column_dimensions["A"].width = 26
    ws.column_dimensions["B"].width = 42
    ws.column_dimensions["C"].width = 14


def main():
    raw_file    = find_raw_file()
    output_file = raw_file.replace("as400_raw_", "as400_analyse_")

    raw = load_raw(raw_file)

    print("\nTransformiere ...")
    tabs_df = t_tabellen(raw)
    stat_df = t_tabellenstatistik(raw)
    idx_df  = t_indizes(raw)

    transforms = {
        "00_Schema_Übersicht":  t_schema_uebersicht(raw),
        "01_Tabellen":          tabs_df,
        "02_Spalten":           t_spalten(raw),
        "03_Views":             t_views(raw),
        "04_Indizes":           idx_df,
        "05_Schlüssel":         t_schluessel(raw),
        "06_Tabellenstatistik": stat_df,
        "07_Spaltenstatistik":  t_spaltenstatistik(raw),
        "08_Programme":         t_programme(raw),
        "09_Wichtigkeit_Score": t_wichtigkeit_score(tabs_df, stat_df, idx_df),
    }

    for name, df in transforms.items():
        status = f"{len(df):>7,} Zeilen" if not df.empty else "leer / View nicht verfügbar"
        print(f"  {name:<28} {status}")

    print(f"\nSchreibe: {output_file} ...")
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for name, df in transforms.items():
            df.to_excel(writer, sheet_name=name[:31], index=False)

    wb = load_workbook(output_file)
    for i, ws_name in enumerate(wb.sheetnames):
        style_sheet(wb[ws_name], TAB_COLORS[i % len(TAB_COLORS)])
    write_info_sheet(wb, output_file, raw_file, transforms)
    wb.save(output_file)

    print(f"✓ Fertig: {output_file}")
    print(f"\nNächster Schritt: as400_phase1_analyse.ipynb öffnen")
    print(f"  Das Notebook sucht automatisch nach as400_analyse_*.xlsx\n")


if __name__ == "__main__":
    main()